# Experiment 2: The Eight Queens Problem
**Course:** CSAI4014 Algorithms for Intelligent Systems and Robotics  
**Student:** Aadi Chopra | SAP ID: 500120163 | Roll No: R2142231902 | Batch: B2 AIML  
**Repository:** https://github.com/aadichopra/aisr-lab-500120163  

## 1. Aim
To solve the eight queens problem using depth-first backtracking search and compare its computational complexity and runtime against generate-and-test across problem sizes $n = 4$ to $10$.


In [ ]:
import os
import time
import math
import itertools
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

os.makedirs('figs', exist_ok=True)

def solve_nqueens_backtracking(n):
    """Backtracking solver with column and diagonal pruning sets."""
    solutions = []
    cols = set(); diag1 = set(); diag2 = set()
    nodes_visited = 0

    def backtrack(row, state):
        nonlocal nodes_visited
        nodes_visited += 1
        if row == n:
            solutions.append(list(state))
            return
        for col in range(n):
            if col not in cols and (row - col) not in diag1 and (row + col) not in diag2:
                cols.add(col); diag1.add(row - col); diag2.add(row + col)
                state.append(col)
                backtrack(row + 1, state)
                state.pop()
                cols.remove(col); diag1.remove(row - col); diag2.remove(row + col)

    start = time.perf_counter()
    backtrack(0, [])
    elapsed = time.perf_counter() - start
    return solutions, nodes_visited, elapsed

def solve_nqueens_generate_test(n):
    """Permutation test baseline (exhaustive n! search)."""
    count = 0
    start = time.perf_counter()
    for perm in itertools.permutations(range(n)):
        d1 = {r - c for r, c in enumerate(perm)}
        d2 = {r + c for r, c in enumerate(perm)}
        if len(d1) == n and len(d2) == n:
            count += 1
    elapsed = time.perf_counter() - start
    return count, math.factorial(n), elapsed

# Run benchmark for n = 4 to 10
n_values = list(range(4, 11))
bt_times, perm_times = [], []
bt_nodes = []

print(f"{'n':>2} | {'Solutions':>9} | {'BT Nodes':>10} | {'BT Time (s)':>12} | {'Perm Time (s)':>14}")
print("-" * 58)
for n in n_values:
    sols, nodes, t_bt = solve_nqueens_backtracking(n)
    bt_times.append(t_bt)
    bt_nodes.append(nodes)
    if n <= 10:
        _, _, t_perm = solve_nqueens_generate_test(n)
    else:
        t_perm = float('nan')
    perm_times.append(t_perm)
    print(f"{n:2d} | {len(sols):9d} | {nodes:10d} | {t_bt:12.6f} | {t_perm:14.6f}")

# Plot Figure 2.1: First solution for n = 8
sols_8, _, _ = solve_nqueens_backtracking(8)
first_sol = sols_8[0] # [0, 4, 7, 5, 2, 6, 1, 3]

fig, ax = plt.subplots(figsize=(4.5, 4.5), dpi=150)
for r in range(8):
    for c in range(8):
        color = '#f0d9b5' if (r + c) % 2 == 0 else '#b58863'
        ax.add_patch(patches.Rectangle((c, 7 - r), 1, 1, facecolor=color, edgecolor='black', linewidth=0.5))

for r, c in enumerate(first_sol):
    ax.text(c + 0.5, 7 - r + 0.5, '♛', fontsize=26, ha='center', va='center', color='black')

ax.set_xlim(0, 8); ax.set_ylim(0, 8)
ax.set_xticks(np.arange(0.5, 8.5, 1)); ax.set_xticklabels(range(8))
ax.set_yticks(np.arange(0.5, 8.5, 1)); ax.set_yticklabels(range(7, -1, -1))
ax.set_xlabel('Column Index'); ax.set_ylabel('Row Index')
ax.set_title('First Solution for 8-Queens Board: [0, 4, 7, 5, 2, 6, 1, 3]', fontsize=10)
plt.tight_layout()
plt.savefig('figs/fig2_1_board8x8.png')
plt.show()

# Plot Figure 2.2: Runtime vs n (log scale)
fig, ax = plt.subplots(figsize=(6.2, 3.8), dpi=150)
ax.plot(n_values, [t * 1000 for t in bt_times], 'o-', color='#1f77b4', linewidth=2, label='Backtracking Search')
ax.plot(n_values, [t * 1000 for t in perm_times], 's--', color='#d62728', linewidth=2, label='Generate-and-Test ($n!$ permutations)')
ax.set_yscale('log')
ax.set_xlabel('Board Size $n$')
ax.set_ylabel('Execution Time (ms, log scale)')
ax.set_title('Execution Time Comparison: Backtracking vs. Generate-and-Test', fontsize=11)
ax.grid(True, which='both', linestyle='--', alpha=0.5)
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('figs/fig2_2_runtime_log.png')
plt.show()
